In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# ============================================================
# Endogenous DC-Copper Spillover Event Detection
# ============================================================
# Event definition:
# Data Center-Copper event =
# 1) SOLVPN -> COPPER raw directional spillover is unusually high
# 2) SOLVPN -> COPPER pairwise dominance is unusually high
# 3) consecutive threshold exceedance points are merged
# 4) event date = peak raw SOLVPN -> COPPER date inside segment
# ============================================================

# ============================================================
# Config
# ============================================================

BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "dc_event_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

SOURCE_VAR = "dlog_SOLVPN"
TARGET_VAR = "dlog_COPPER"

# Main thresholds
RAW_Q = 0.95
NET_Q = 0.80

# Optional absolute floor
USE_RAW_ABS_FLOOR = True
RAW_ABS_FLOOR = 15.0

# Segment filtering
MIN_SEGMENT_LENGTH = 2
MIN_EVENT_GAP = 10

# Event windows
WINDOWS = [5, 10, 20]

EPS = 1e-12


# ============================================================
# Helpers
# ============================================================

def infer_gfevd_tnn(arr: np.ndarray) -> np.ndarray:
    if arr.ndim != 3:
        raise ValueError(f"gfevd_all must be 3D. Got shape={arr.shape}")

    if arr.shape[1] == arr.shape[2]:
        return arr.astype(float, copy=True)

    if arr.shape[0] == arr.shape[1]:
        return np.transpose(arr, (2, 0, 1)).astype(float, copy=True)

    raise ValueError(f"Cannot infer GFEVD shape. Got shape={arr.shape}")


def normalize_gfevd_to_percent(gfevd_tnn: np.ndarray) -> np.ndarray:
    arr = np.asarray(gfevd_tnn, dtype=float).copy()

    if not np.all(np.isfinite(arr)):
        raise ValueError("GFEVD contains NaN or inf.")

    if arr.min() < -1e-10:
        raise ValueError(f"GFEVD contains negative values. min={arr.min()}")

    arr[arr < 0] = 0.0

    row_sum = arr.sum(axis=2, keepdims=True)

    if np.any(row_sum <= EPS):
        bad = np.argwhere(row_sum.squeeze(-1) <= EPS)
        raise ValueError(f"Zero row sum detected. First bad index={bad[0].tolist()}")

    theta_pct = arr / row_sum * 100.0

    max_err = np.max(np.abs(theta_pct.sum(axis=2) - 100.0))
    if max_err > 1e-8:
        raise RuntimeError(f"Row normalization failed. max_err={max_err}")

    return theta_pct


def load_dates(target_len: int):
    for fp in DATE_FILE_CANDIDATES:
        if not fp.exists():
            continue

        try:
            df = pd.read_csv(fp)

            date_col = None
            for c in ["Date", "date", "DATE"]:
                if c in df.columns:
                    date_col = c
                    break

            if date_col is None:
                continue

            dates = pd.to_datetime(df[date_col], errors="coerce")
            dates = dates.dropna().reset_index(drop=True)

            if len(dates) < target_len:
                print(f"[WARN] {fp} has {len(dates)} valid dates, target={target_len}. Skipped.")
                continue

            return dates.iloc[-target_len:].reset_index(drop=True)

        except Exception as e:
            print(f"[WARN] Failed to read dates from {fp}: {e}")

    return None


def compute_pairwise_net(theta_pct: np.ndarray) -> np.ndarray:
    """
    theta[target, source] = source -> target spillover
    pairwise_net[source, target] = theta[target, source] - theta[source, target]
    """
    return theta_pct.T - theta_pct


def merge_event_segments(mask: np.ndarray, raw_signal: np.ndarray, min_segment_length: int = 1):
    """
    Consecutive True regions are merged.
    Event date = index where raw_signal is maximum inside each segment.
    """
    segments = []
    in_seg = False
    start = None

    for i, flag in enumerate(mask):
        if flag and not in_seg:
            start = i
            in_seg = True
        elif (not flag) and in_seg:
            end = i - 1
            segments.append((start, end))
            in_seg = False

    if in_seg:
        segments.append((start, len(mask) - 1))

    events = []

    for start, end in segments:
        seg_len = end - start + 1

        if seg_len < min_segment_length:
            continue

        local = raw_signal[start:end + 1]
        peak_idx = start + int(np.argmax(local))

        events.append({
            "segment_start_idx": start,
            "segment_end_idx": end,
            "event_idx": peak_idx,
            "segment_length": seg_len,
        })

    return events


def enforce_min_event_gap(events: list, raw_signal: np.ndarray, min_gap: int):
    """
    If detected events are too close, keep the stronger one by raw_signal peak.
    """
    if len(events) == 0:
        return []

    events_sorted = sorted(events, key=lambda x: x["event_idx"])
    kept = [events_sorted[0]]

    for ev in events_sorted[1:]:
        prev = kept[-1]

        if ev["event_idx"] - prev["event_idx"] < min_gap:
            if raw_signal[ev["event_idx"]] > raw_signal[prev["event_idx"]]:
                kept[-1] = ev
        else:
            kept.append(ev)

    return kept


def window_stats(series: np.ndarray, event_idx: int, w: int):
    """
    pre  = [t0-w, t0-1]
    post = [t0+1, t0+w]
    """
    n = len(series)

    pre_start = max(0, event_idx - w)
    pre_end = event_idx

    post_start = event_idx + 1
    post_end = min(n, event_idx + w + 1)

    pre = series[pre_start:pre_end]
    post = series[post_start:post_end]

    pre_mean = np.nan if len(pre) == 0 else float(np.mean(pre))
    post_mean = np.nan if len(post) == 0 else float(np.mean(post))

    if np.isnan(pre_mean) or np.isnan(post_mean):
        delta = np.nan
    else:
        delta = post_mean - pre_mean

    return pre_mean, post_mean, delta


def after_event_features(series: np.ndarray, event_idx: int, w: int):
    """
    Post-event features using [t0, t0+w].
    """
    n = len(series)
    start = event_idx
    end = min(n, event_idx + w + 1)

    seg = series[start:end]

    if len(seg) == 0:
        return np.nan, np.nan, np.nan

    peak = float(np.max(seg))
    peak_horizon = int(np.argmax(seg))
    auc = float(np.sum(seg))

    return peak, peak_horizon, auc


def classify_event(delta_raw, delta_rev, delta_net):
    """
    Window-level classification.
    """
    if pd.isna(delta_raw) or pd.isna(delta_rev) or pd.isna(delta_net):
        return "Weak/Neutral"

    if delta_raw > 0 and delta_net > 0:
        return "DC-driven"

    if delta_rev > 0 and delta_net < 0:
        return "Copper-driven"

    if delta_raw > 0 and delta_rev > 0:
        return "Coupled"

    return "Weak/Neutral"


# ============================================================
# 1. Load GFEVD
# ============================================================

if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE.resolve()}")

raw_gfevd = np.load(GFEVD_FILE)
gfevd_tnn = infer_gfevd_tnn(raw_gfevd)
theta_pct_all = normalize_gfevd_to_percent(gfevd_tnn)

T_eff, N, N2 = theta_pct_all.shape

if N != N2:
    raise ValueError(f"GFEVD last two dims must be square. Got {theta_pct_all.shape}")

if len(VAR_NAMES) != N:
    raise ValueError(f"len(VAR_NAMES)={len(VAR_NAMES)} does not match N={N}")

if SOURCE_VAR not in VAR_NAMES:
    raise ValueError(f"{SOURCE_VAR} not found in VAR_NAMES")

if TARGET_VAR not in VAR_NAMES:
    raise ValueError(f"{TARGET_VAR} not found in VAR_NAMES")

source_idx = VAR_NAMES.index(SOURCE_VAR)
target_idx = VAR_NAMES.index(TARGET_VAR)

dates = load_dates(T_eff)

if dates is None:
    time_index = pd.Series(np.arange(T_eff))
    time_col = "t"
else:
    time_index = dates
    time_col = "Date"

print(f"[INFO] Loaded GFEVD: {theta_pct_all.shape}")
print(f"[INFO] Source: {SOURCE_VAR}")
print(f"[INFO] Target: {TARGET_VAR}")
print(f"[INFO] Time index: {time_col}")


# ============================================================
# 2. Build spillover signals
# ============================================================

raw_source_to_target = theta_pct_all[:, target_idx, source_idx]
raw_target_to_source = theta_pct_all[:, source_idx, target_idx]

pairwise_net_all = np.zeros((T_eff, N, N), dtype=float)

for t in range(T_eff):
    pairwise_net_all[t] = compute_pairwise_net(theta_pct_all[t])

pairwise_net_source_to_target = pairwise_net_all[:, source_idx, target_idx]

source_to_all = (
    theta_pct_all[:, :, source_idx].sum(axis=1)
    - theta_pct_all[:, source_idx, source_idx]
)


# ============================================================
# 3. Threshold-based endogenous event detection
# ============================================================

raw_q_threshold = float(np.quantile(raw_source_to_target, RAW_Q))
net_q_threshold = float(np.quantile(pairwise_net_source_to_target, NET_Q))

raw_threshold = max(raw_q_threshold, RAW_ABS_FLOOR) if USE_RAW_ABS_FLOOR else raw_q_threshold

event_mask_raw = raw_source_to_target >= raw_threshold
event_mask_net = pairwise_net_source_to_target >= net_q_threshold

event_mask = event_mask_raw & event_mask_net

events_raw = merge_event_segments(
    event_mask,
    raw_source_to_target,
    min_segment_length=MIN_SEGMENT_LENGTH,
)

events = enforce_min_event_gap(
    events_raw,
    raw_source_to_target,
    min_gap=MIN_EVENT_GAP,
)

print(f"[INFO] raw Q{int(RAW_Q * 100)} threshold = {raw_q_threshold:.4f}")
print(f"[INFO] raw final threshold = {raw_threshold:.4f}")
print(f"[INFO] net Q{int(NET_Q * 100)} threshold = {net_q_threshold:.4f}")
print(f"[INFO] initial event segments = {len(events_raw)}")
print(f"[INFO] final events after min-gap filter = {len(events)}")


# ============================================================
# 4. Save full signal time series
# ============================================================

df_signal = pd.DataFrame({
    time_col: time_index,
    "raw_SOLVPN_to_COPPER": raw_source_to_target,
    "raw_COPPER_to_SOLVPN": raw_target_to_source,
    "pairwise_net_SOLVPN_to_COPPER": pairwise_net_source_to_target,
    "SOLVPN_to_ALL": source_to_all,
    "event_mask_raw": event_mask_raw.astype(int),
    "event_mask_net": event_mask_net.astype(int),
    "event_mask_final": event_mask.astype(int),
})

OUT_SIGNAL = OUT_DIR / "dc_copper_spillover_signals.csv"
df_signal.to_csv(OUT_SIGNAL, index=False, encoding="utf-8-sig")

print(f"[INFO] Saved signal file: {OUT_SIGNAL}")


# ============================================================
# 5. Event-level feature extraction
# ============================================================

event_rows = []

for k, ev in enumerate(events, start=1):
    event_idx = ev["event_idx"]
    seg_start = ev["segment_start_idx"]
    seg_end = ev["segment_end_idx"]

    row = {
        "EventID": k,
        time_col: time_index.iloc[event_idx],
        "event_idx": event_idx,
        "segment_start_idx": seg_start,
        "segment_end_idx": seg_end,
        "segment_length": ev["segment_length"],
        "raw_SOLVPN_to_COPPER_at_event": raw_source_to_target[event_idx],
        "raw_COPPER_to_SOLVPN_at_event": raw_target_to_source[event_idx],
        "pairwise_net_SOLVPN_to_COPPER_at_event": pairwise_net_source_to_target[event_idx],
        "SOLVPN_to_ALL_at_event": source_to_all[event_idx],
    }

    for w in WINDOWS:
        pre_raw, post_raw, delta_raw = window_stats(raw_source_to_target, event_idx, w)
        pre_rev, post_rev, delta_rev = window_stats(raw_target_to_source, event_idx, w)
        pre_net, post_net, delta_net = window_stats(pairwise_net_source_to_target, event_idx, w)
        pre_all, post_all, delta_all = window_stats(source_to_all, event_idx, w)

        peak_raw, peak_h_raw, auc_raw = after_event_features(raw_source_to_target, event_idx, w)
        peak_rev, peak_h_rev, auc_rev = after_event_features(raw_target_to_source, event_idx, w)
        peak_net, peak_h_net, auc_net = after_event_features(pairwise_net_source_to_target, event_idx, w)

        row[f"W{w}_pre_raw_SOLVPN_to_COPPER"] = pre_raw
        row[f"W{w}_post_raw_SOLVPN_to_COPPER"] = post_raw
        row[f"W{w}_delta_raw_SOLVPN_to_COPPER"] = delta_raw

        row[f"W{w}_pre_raw_COPPER_to_SOLVPN"] = pre_rev
        row[f"W{w}_post_raw_COPPER_to_SOLVPN"] = post_rev
        row[f"W{w}_delta_raw_COPPER_to_SOLVPN"] = delta_rev

        row[f"W{w}_pre_pairwise_net"] = pre_net
        row[f"W{w}_post_pairwise_net"] = post_net
        row[f"W{w}_delta_pairwise_net"] = delta_net

        row[f"W{w}_pre_SOLVPN_to_ALL"] = pre_all
        row[f"W{w}_post_SOLVPN_to_ALL"] = post_all
        row[f"W{w}_delta_SOLVPN_to_ALL"] = delta_all

        row[f"W{w}_peak_raw_SOLVPN_to_COPPER"] = peak_raw
        row[f"W{w}_peak_horizon_raw_SOLVPN_to_COPPER"] = peak_h_raw
        row[f"W{w}_auc_raw_SOLVPN_to_COPPER"] = auc_raw

        row[f"W{w}_peak_raw_COPPER_to_SOLVPN"] = peak_rev
        row[f"W{w}_peak_horizon_raw_COPPER_to_SOLVPN"] = peak_h_rev
        row[f"W{w}_auc_raw_COPPER_to_SOLVPN"] = auc_rev

        row[f"W{w}_peak_pairwise_net"] = peak_net
        row[f"W{w}_peak_horizon_pairwise_net"] = peak_h_net
        row[f"W{w}_auc_pairwise_net"] = auc_net

        row[f"W{w}_event_type"] = classify_event(delta_raw, delta_rev, delta_net)

    event_rows.append(row)

df_events = pd.DataFrame(event_rows)

OUT_EVENTS = OUT_DIR / "dc_copper_events.csv"
df_events.to_csv(OUT_EVENTS, index=False, encoding="utf-8-sig")

print(f"[INFO] Saved event file: {OUT_EVENTS}")


# ============================================================
# 6. Window-level summary
# ============================================================

summary_rows = []

for w in WINDOWS:
    if len(df_events) == 0:
        continue

    type_col = f"W{w}_event_type"
    counts = df_events[type_col].value_counts().to_dict()

    summary_rows.append({
        "Window": w,
        "N_events": len(df_events),
        "Mean_delta_raw_SOLVPN_to_COPPER": df_events[f"W{w}_delta_raw_SOLVPN_to_COPPER"].mean(),
        "Median_delta_raw_SOLVPN_to_COPPER": df_events[f"W{w}_delta_raw_SOLVPN_to_COPPER"].median(),
        "Mean_delta_raw_COPPER_to_SOLVPN": df_events[f"W{w}_delta_raw_COPPER_to_SOLVPN"].mean(),
        "Median_delta_raw_COPPER_to_SOLVPN": df_events[f"W{w}_delta_raw_COPPER_to_SOLVPN"].median(),
        "Mean_delta_pairwise_net": df_events[f"W{w}_delta_pairwise_net"].mean(),
        "Median_delta_pairwise_net": df_events[f"W{w}_delta_pairwise_net"].median(),
        "Mean_delta_SOLVPN_to_ALL": df_events[f"W{w}_delta_SOLVPN_to_ALL"].mean(),
        "Median_delta_SOLVPN_to_ALL": df_events[f"W{w}_delta_SOLVPN_to_ALL"].median(),
        "N_DC_driven": counts.get("DC-driven", 0),
        "N_Copper_driven": counts.get("Copper-driven", 0),
        "N_Coupled": counts.get("Coupled", 0),
        "N_Weak_Neutral": counts.get("Weak/Neutral", 0),
    })

df_summary = pd.DataFrame(summary_rows)

OUT_SUMMARY = OUT_DIR / "dc_copper_event_window_summary.csv"
df_summary.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

print(f"[INFO] Saved summary file: {OUT_SUMMARY}")


# ============================================================
# 7. Robustness thresholds summary
# ============================================================

robust_rows = []

raw_q_list = [0.90, 0.95]
net_q_list = [0.75, 0.80, 0.85]

for rq in raw_q_list:
    for nq in net_q_list:
        rq_thr = float(np.quantile(raw_source_to_target, rq))
        nq_thr = float(np.quantile(pairwise_net_source_to_target, nq))

        rq_final = max(rq_thr, RAW_ABS_FLOOR) if USE_RAW_ABS_FLOOR else rq_thr

        mask = (raw_source_to_target >= rq_final) & (pairwise_net_source_to_target >= nq_thr)

        ev0 = merge_event_segments(
            mask,
            raw_source_to_target,
            min_segment_length=MIN_SEGMENT_LENGTH,
        )
        ev1 = enforce_min_event_gap(
            ev0,
            raw_source_to_target,
            min_gap=MIN_EVENT_GAP,
        )

        robust_rows.append({
            "RAW_Q": rq,
            "NET_Q": nq,
            "raw_threshold": rq_final,
            "net_threshold": nq_thr,
            "N_initial_segments": len(ev0),
            "N_final_events": len(ev1),
        })

df_robust = pd.DataFrame(robust_rows)

OUT_ROBUST = OUT_DIR / "dc_copper_threshold_robustness.csv"
df_robust.to_csv(OUT_ROBUST, index=False, encoding="utf-8-sig")

print(f"[INFO] Saved robustness file: {OUT_ROBUST}")


# ============================================================
# 8. Plots
# ============================================================

x = pd.to_datetime(df_signal[time_col]) if time_col == "Date" else df_signal[time_col]

event_x = []
for ev in events:
    idx = ev["event_idx"]
    event_x.append(time_index.iloc[idx])

# Plot 1: Raw SOLVPN -> COPPER
plt.figure(figsize=(14, 5))
plt.plot(x, raw_source_to_target, linewidth=1.2, label="SOLVPN → COPPER raw spillover")
plt.axhline(raw_threshold, linestyle="--", linewidth=1.0, label="Raw threshold")

for ex in event_x:
    plt.axvline(ex, linestyle=":", linewidth=0.9)

plt.title("Endogenous DC-Copper Event Detection: Raw SOLVPN → COPPER")
plt.xlabel(time_col)
plt.ylabel("Directional Spillover (%)")
plt.legend()
plt.tight_layout()

OUT_PLOT_RAW = OUT_DIR / "event_detection_raw_solvpn_to_copper.png"
plt.savefig(OUT_PLOT_RAW, dpi=300, bbox_inches="tight")
plt.close()

# Plot 2: Pairwise net
plt.figure(figsize=(14, 5))
plt.plot(x, pairwise_net_source_to_target, linewidth=1.2, label="Pairwise Net: SOLVPN → COPPER dominance")
plt.axhline(0.0, linestyle="-", linewidth=0.8)
plt.axhline(net_q_threshold, linestyle="--", linewidth=1.0, label="Net threshold")

for ex in event_x:
    plt.axvline(ex, linestyle=":", linewidth=0.9)

plt.title("Endogenous DC-Copper Event Detection: Pairwise Net Dominance")
plt.xlabel(time_col)
plt.ylabel("Pairwise NET (%)")
plt.legend()
plt.tight_layout()

OUT_PLOT_NET = OUT_DIR / "event_detection_pairwise_net.png"
plt.savefig(OUT_PLOT_NET, dpi=300, bbox_inches="tight")
plt.close()

# Plot 3: Combined
plt.figure(figsize=(14, 6))
plt.plot(x, raw_source_to_target, linewidth=1.2, label="Raw SOLVPN → COPPER")
plt.plot(x, pairwise_net_source_to_target, linewidth=1.0, label="Pairwise Net SOLVPN → COPPER")
plt.axhline(0.0, linestyle="-", linewidth=0.8)

for ex in event_x:
    plt.axvline(ex, linestyle=":", linewidth=0.9)

plt.title("Endogenous DC-Copper Events: Raw Spillover and Pairwise Net")
plt.xlabel(time_col)
plt.ylabel("Spillover / NET (%)")
plt.legend()
plt.tight_layout()

OUT_PLOT_COMBINED = OUT_DIR / "event_detection_combined.png"
plt.savefig(OUT_PLOT_COMBINED, dpi=300, bbox_inches="tight")
plt.close()

print("[INFO] Saved plots:")
print(f"  - {OUT_PLOT_RAW}")
print(f"  - {OUT_PLOT_NET}")
print(f"  - {OUT_PLOT_COMBINED}")


# ============================================================
# 9. Print preview
# ============================================================

print("\n[INFO] Event preview")
if len(df_events) > 0:
    preview_cols = [
        "EventID",
        time_col,
        "event_idx",
        "segment_length",
        "raw_SOLVPN_to_COPPER_at_event",
        "raw_COPPER_to_SOLVPN_at_event",
        "pairwise_net_SOLVPN_to_COPPER_at_event",
    ]

    for w in WINDOWS:
        preview_cols += [
            f"W{w}_delta_raw_SOLVPN_to_COPPER",
            f"W{w}_delta_pairwise_net",
            f"W{w}_event_type",
        ]

    print(df_events[preview_cols].to_string(index=False))
else:
    print("No events detected. Consider lowering RAW_Q, NET_Q, RAW_ABS_FLOOR, or MIN_SEGMENT_LENGTH.")

print("\n[INFO] Window summary")
if len(df_summary) > 0:
    print(df_summary.to_string(index=False))
else:
    print("No summary available because no events were detected.")

print("\n[INFO] Threshold robustness")
print(df_robust.to_string(index=False))

[INFO] Loaded GFEVD: (1031, 9, 9)
[INFO] Source: dlog_SOLVPN
[INFO] Target: dlog_COPPER
[INFO] Time index: t
[INFO] raw Q95 threshold = 16.0550
[INFO] raw final threshold = 16.0550
[INFO] net Q80 threshold = 3.6312
[INFO] initial event segments = 4
[INFO] final events after min-gap filter = 4
[INFO] Saved signal file: dc_event_output\dc_copper_spillover_signals.csv
[INFO] Saved event file: dc_event_output\dc_copper_events.csv
[INFO] Saved summary file: dc_event_output\dc_copper_event_window_summary.csv
[INFO] Saved robustness file: dc_event_output\dc_copper_threshold_robustness.csv
[INFO] Saved plots:
  - dc_event_output\event_detection_raw_solvpn_to_copper.png
  - dc_event_output\event_detection_pairwise_net.png
  - dc_event_output\event_detection_combined.png

[INFO] Event preview
 EventID   t  event_idx  segment_length  raw_SOLVPN_to_COPPER_at_event  raw_COPPER_to_SOLVPN_at_event  pairwise_net_SOLVPN_to_COPPER_at_event  W5_delta_raw_SOLVPN_to_COPPER  W5_delta_pairwise_net W5_event_t